In [1]:
import pandas as pd
import numpy as np

## Importar el Dataset y Revisar Tipos de Datos

In [2]:
df = pd.read_csv("evaluaciones_docentes.csv", sep=",", encoding="utf-8", header="infer", dtype={
    "id_docente": "category", "tendencia_desempeno": "category", "comentario": "category", "asignatura": "category"
}, converters={
    # Hay semestres donde el docente deja de dictar una materia y necesita agregarse el semestre artificialmente.
    # Por tanto, se agrega esta transformación para más adelante apoyar a la completacion de ceros en semestres faltantes.
    "semestre": lambda s: pd.Period(s.strip().replace("-1", "Q1").replace("-2", "Q3"), freq="2Q-DEC")
})
df = df.convert_dtypes()
df.dtypes

id_docente                   category
asignatura                   category
semestre               period[2Q-DEC]
numero_estudiantes              Int64
puntaje_claridad              Float64
puntaje_metodologia           Float64
puntaje_evaluacion            Float64
comentario                   category
tendencia_desempeno          category
dtype: object

## Funciones de Utilería para Semestres

In [3]:
# Función para devolver la representación de un periodo a formato año-semestre (para visualizaciones, resultados finales, etc)
def period_to_string(periodo: pd.Period):
    return f"{periodo.year}-{(periodo.quarter+1)//2}"

# Más adelante para completar semestres, solo sería revisar el número de periodos entre 2 muestras.
def numero_de_periodos_entre_semestres(semestre_final: pd.Period, semestre_inicial: pd.Period):
    if pd.isna(semestre_final) or pd.isna(semestre_inicial):
        return 0
    return 1 + (semestre_final - semestre_inicial).n//2

## Estructuración de Consultas para Objetivos Especificos

### Grupos que cada Docente maneja para cada Materia

In [4]:
grupos_que_el_docente_maneja_para_cada_materia = (
    df.groupby(["id_docente", "semestre", "asignatura"], as_index=False)
    .agg(
        Cantidad_de_Grupos=("semestre", "size"), # Counts rows per group
        numero_estudiantes=("numero_estudiantes", "sum"),
        puntaje_claridad=("puntaje_claridad", "mean"),
        puntaje_metodologia=("puntaje_metodologia", "mean"),
        puntaje_evaluacion=("puntaje_evaluacion", "mean"),
    )
    .rename(columns={"Cantidad_de_Grupos": "Cantidad de Grupos"})
    .sort_values(by=["id_docente", "asignatura", "semestre"], ascending=[True, True, True])
)

grupos_que_el_docente_maneja_para_cada_materia["Cantidad de Grupos"] = grupos_que_el_docente_maneja_para_cada_materia["Cantidad de Grupos"].astype("Int64")
grupos_que_el_docente_maneja_para_cada_materia["numero_estudiantes"] = grupos_que_el_docente_maneja_para_cada_materia["numero_estudiantes"].astype("Int64")

grupos_que_el_docente_maneja_para_cada_materia

,id_docente,semestre,asignatura,Cantidad de Grupos,numero_estudiantes,puntaje_claridad,puntaje_metodologia,puntaje_evaluacion
0,Docente_1,2020Q1,Bases de Datos,1,17,4.3,4.8,3.6
5,Docente_1,2020Q3,Bases de Datos,2,61,2.75,2.85,2.7
11,Docente_1,2021Q1,Bases de Datos,1,19,4.0,4.5,3.2
16,Docente_1,2021Q3,Bases de Datos,1,16,2.7,2.7,1.0
27,Docente_1,2023Q1,Bases de Datos,1,33,3.9,3.6,3.6
...,...,...,...,...,...,...,...,...
1840,Docente_9,2020Q3,Ética Profesional,2,48,3.15,3.45,3.0
1849,Docente_9,2021Q3,Ética Profesional,1,41,3.3,2.5,2.3
1855,Docente_9,2022Q1,Ética Profesional,2,34,3.3,4.0,3.5
1863,Docente_9,2023Q1,Ética Profesional,1,26,4.5,4.8,4.4


### Completación de Semestres Faltantes y Métricas de Semestres

In [5]:
def completar_semestres(muestras_docente):

    muestras = muestras_docente.copy(deep=True)
    muestras.reset_index(inplace=True)

    # Campos auxiliares para completación de semestres
    muestras["semestre previo"] = muestras.semestre.shift(1)
    muestras["Numero de Periodos entre Semestres"] = muestras.apply(lambda row: numero_de_periodos_entre_semestres(row["semestre"], row["semestre previo"]), axis=1)

    # Util para darle más contexto al área usuaria y a modelos de inteligencia artificial
    muestras["Semestres Desde Ultima Calificación"] = pd.NA
    muestras.iloc[1:, muestras.columns.get_loc("Semestres Desde Ultima Calificación")] = 1

    gaps = muestras[muestras["Numero de Periodos entre Semestres"] > 2].index.tolist()
    if len(gaps) == 0:
        return muestras_docente

    fields = [
        "semestre",
        "Numero de Periodos entre Semestres",
        "semestre previo",
        "Cantidad de Grupos",
        "Semestres Desde Ultima Calificación",
        "numero_estudiantes",
        "puntaje_claridad",
        "puntaje_metodologia",
        "puntaje_evaluacion",
    ]

    # Limita hasta cuantos semestres puedo completar hacia atrás, de momento deja hasta 1000.
    step = 1e-3
    for idx_grupito in gaps:
        semestres_faltantes = muestras.loc[idx_grupito]["Numero de Periodos entre Semestres"] - 2
        muestras.loc[idx_grupito, ["Semestres Desde Ultima Calificación"]] = [muestras.loc[idx_grupito]["Numero de Periodos entre Semestres"] - 1]

        idx_semestre = idx_grupito
        for sem_faltante in range(semestres_faltantes):
            # Clonado y sobre-escritura de campos, asignando el 0 en Cantidad de Grupos y fijando la Última Vez Que Fué Calificado.
            muestras.loc[idx_semestre - step] = muestras.loc[idx_semestre].copy()
            muestras.loc[idx_semestre - step, fields] = [
                muestras.loc[idx_semestre].semestre - 1,
                0,
                pd.NaT,
                pd.NA,
                semestres_faltantes - sem_faltante,
                pd.NA,
                pd.NA,
                pd.NA,
                pd.NA,
            ]
            idx_semestre = idx_semestre - step

    # Estos 2 pasos ordenan correctamente lo que se agregó
    muestras.sort_index(inplace=True)
    muestras.reset_index(drop=True, inplace=True)

    # Ya podemos eliminar los campos auxiliares
    muestras.drop(columns=["semestre previo", "Numero de Periodos entre Semestres"], inplace=True)
    muestras.set_index(["id_docente", "asignatura"], inplace=True)

    muestras["Semestres Desde Ultima Calificación"] = muestras["Semestres Desde Ultima Calificación"].astype("Int64")
    muestras["Cantidad de Grupos"] = muestras["Cantidad de Grupos"].astype("Int64")

    return muestras

In [6]:
# El resultado es simplemente que se agreguen filas para cada semestre que le haga falta al docente para una materia particular.
# Las que se hayan agregado artificialmente van a tener un valor de NA en el campo "Cantidad de Grupos".
grupos_que_el_docente_maneja_para_cada_materia = (
    grupos_que_el_docente_maneja_para_cada_materia
        .set_index(["id_docente", "asignatura"], append=False)
        .groupby(level=["id_docente", "asignatura"], group_keys=False)
        .apply(completar_semestres, include_groups=False)
        .reset_index()
)
grupos_que_el_docente_maneja_para_cada_materia

,id_docente,asignatura,semestre,Cantidad de Grupos,numero_estudiantes,puntaje_claridad,puntaje_metodologia,puntaje_evaluacion,Semestres Desde Ultima Calificación
0,Docente_1,Bases de Datos,2020Q1,1,17,4.3,4.8,3.6,<NA>
1,Docente_1,Bases de Datos,2020Q3,2,61,2.75,2.85,2.7,1
2,Docente_1,Bases de Datos,2021Q1,1,19,4.0,4.5,3.2,1
3,Docente_1,Bases de Datos,2021Q3,1,16,2.7,2.7,1.0,1
4,Docente_1,Bases de Datos,2022Q1,<NA>,<NA>,<NA>,<NA>,<NA>,1
...,...,...,...,...,...,...,...,...,...
2474,Docente_9,Ética Profesional,2021Q3,1,41,3.3,2.5,2.3,2
2475,Docente_9,Ética Profesional,2022Q1,2,34,3.3,4.0,3.5,1
2476,Docente_9,Ética Profesional,2022Q3,<NA>,<NA>,<NA>,<NA>,<NA>,1
2477,Docente_9,Ética Profesional,2023Q1,1,26,4.5,4.8,4.4,2


### Métrica de Rotación Docente para cada Materia

In [7]:
def agregar_metricas(muestras_docente):
    muestras = muestras_docente.copy(deep=True)

    muestras.reset_index(inplace=True)

    muestras["Diferencia en Cantidad de Grupos con Semestre Anterior"] = (
        muestras["Cantidad de Grupos"]
        - muestras["Cantidad de Grupos"].shift(1)
    )

    muestras["Diferencia en Semestres Desde Ultima Calificación"] = (
        muestras["Semestres Desde Ultima Calificación"]
        - muestras["Semestres Desde Ultima Calificación"].shift(1)
    )

    muestras["Reingreso"] = (
        ( ~ muestras["Cantidad de Grupos"].isna() )
        & muestras["Diferencia en Cantidad de Grupos con Semestre Anterior"].isna()
        & (muestras["Diferencia en Semestres Desde Ultima Calificación"] > 0)
    )

    muestras["Reingreso"] = muestras.apply(lambda row: pd.NA if row["Reingreso"] is False and pd.isna(row["Cantidad de Grupos"]) else row["Reingreso"], axis=1)

    reingresos_acumulados = muestras["Reingreso"].cumsum()
    indice_rotacion = reingresos_acumulados / (reingresos_acumulados.index + 1)

    muestras["Indice de Reingreso"] = indice_rotacion

    muestras.drop(columns=["Diferencia en Cantidad de Grupos con Semestre Anterior", "Diferencia en Semestres Desde Ultima Calificación"], inplace=True)

    muestras.set_index(["id_docente", "asignatura"], inplace=True)

    return muestras

In [8]:
# Se genera el KPI de rotación necesario para el área usuaria y los modelos de inteligencia artificial.
grupos_que_el_docente_maneja_para_cada_materia = (
    grupos_que_el_docente_maneja_para_cada_materia
    .set_index(["id_docente", "asignatura"], append=False)
    .groupby(level=["id_docente", "asignatura"], group_keys=False)
    .apply(agregar_metricas, include_groups=False)
    .reset_index()
)
grupos_que_el_docente_maneja_para_cada_materia

,id_docente,asignatura,semestre,Cantidad de Grupos,numero_estudiantes,puntaje_claridad,puntaje_metodologia,puntaje_evaluacion,Semestres Desde Ultima Calificación,Reingreso,Indice de Reingreso
0,Docente_1,Bases de Datos,2020Q1,1,17,4.3,4.8,3.6,<NA>,<NA>,NaN
1,Docente_1,Bases de Datos,2020Q3,2,61,2.75,2.85,2.7,1,False,0.0
2,Docente_1,Bases de Datos,2021Q1,1,19,4.0,4.5,3.2,1,False,0.0
3,Docente_1,Bases de Datos,2021Q3,1,16,2.7,2.7,1.0,1,False,0.0
4,Docente_1,Bases de Datos,2022Q1,<NA>,<NA>,<NA>,<NA>,<NA>,1,<NA>,NaN
...,...,...,...,...,...,...,...,...,...,...,...
2474,Docente_9,Ética Profesional,2021Q3,1,41,3.3,2.5,2.3,2,True,0.25
2475,Docente_9,Ética Profesional,2022Q1,2,34,3.3,4.0,3.5,1,False,0.2
2476,Docente_9,Ética Profesional,2022Q3,<NA>,<NA>,<NA>,<NA>,<NA>,1,<NA>,NaN
2477,Docente_9,Ética Profesional,2023Q1,1,26,4.5,4.8,4.4,2,True,0.285714


In [9]:
grupos_que_el_docente_maneja_para_cada_materia.dropna(subset=["Reingreso"])

,id_docente,asignatura,semestre,Cantidad de Grupos,numero_estudiantes,puntaje_claridad,puntaje_metodologia,puntaje_evaluacion,Semestres Desde Ultima Calificación,Reingreso,Indice de Reingreso
1,Docente_1,Bases de Datos,2020Q3,2,61,2.75,2.85,2.7,1,False,0.0
2,Docente_1,Bases de Datos,2021Q1,1,19,4.0,4.5,3.2,1,False,0.0
3,Docente_1,Bases de Datos,2021Q3,1,16,2.7,2.7,1.0,1,False,0.0
6,Docente_1,Bases de Datos,2023Q1,1,33,3.9,3.6,3.6,3,True,0.142857
7,Docente_1,Bases de Datos,2023Q3,1,27,2.7,2.7,3.4,1,False,0.125
...,...,...,...,...,...,...,...,...,...,...,...
2472,Docente_9,Ética Profesional,2020Q3,2,48,3.15,3.45,3.0,1,False,0.0
2474,Docente_9,Ética Profesional,2021Q3,1,41,3.3,2.5,2.3,2,True,0.25
2475,Docente_9,Ética Profesional,2022Q1,2,34,3.3,4.0,3.5,1,False,0.2
2477,Docente_9,Ética Profesional,2023Q1,1,26,4.5,4.8,4.4,2,True,0.285714


### Número de Materias y Grupos Manejados por Cada Docente - Histórico

In [10]:
cantidad_de_materias_que_el_docente_maneja_por_cada_semestre = (
    df.groupby(["id_docente", "semestre"], as_index=False)
    .agg(
        Cantidad_de_Materias=("asignatura", "nunique"),
        Cantidad_de_Grupos=("semestre", "size"), # Counts rows per group
        numero_estudiantes=("numero_estudiantes", "sum"),
        # puntaje_claridad=("puntaje_claridad", "mean"),
        # puntaje_metodologia=("puntaje_metodologia", "mean"),
        # puntaje_evaluacion=("puntaje_evaluacion", "mean"),
    )
    .rename(columns={"Cantidad_de_Grupos": "Cantidad de Grupos", "Cantidad_de_Materias": "Cantidad de Materias"})
    .sort_values(by=["id_docente", "semestre"], ascending=[True, True])
)

cantidad_de_materias_que_el_docente_maneja_por_cada_semestre["Cantidad de Materias"] = cantidad_de_materias_que_el_docente_maneja_por_cada_semestre["Cantidad de Materias"].astype("Int64")
cantidad_de_materias_que_el_docente_maneja_por_cada_semestre["Cantidad de Grupos"] = cantidad_de_materias_que_el_docente_maneja_por_cada_semestre["Cantidad de Grupos"].astype("Int64")
cantidad_de_materias_que_el_docente_maneja_por_cada_semestre["numero_estudiantes"] = cantidad_de_materias_que_el_docente_maneja_por_cada_semestre["numero_estudiantes"].astype("Int64")


cantidad_de_materias_que_el_docente_maneja_por_cada_semestre

,id_docente,semestre,Cantidad de Materias,Cantidad de Grupos,numero_estudiantes
0,Docente_1,2020Q1,5,7,205
1,Docente_1,2020Q3,6,9,303
2,Docente_1,2021Q1,5,9,208
3,Docente_1,2021Q3,5,7,182
4,Docente_1,2022Q1,3,6,195
...,...,...,...,...,...
395,Docente_9,2021Q3,5,6,199
396,Docente_9,2022Q1,6,9,305
397,Docente_9,2022Q3,5,8,281
398,Docente_9,2023Q1,3,4,101


### Métrica de Carga Académica

### Metricas

In [11]:
def agregar_metrica_carga_academica(muestras_docente):
    muestras = muestras_docente.copy(deep=True)
    muestras.reset_index(inplace=True) # drop multi-index

    muestras.reset_index(inplace=True)  # creates 'index' column with original positions
    muestras["Max Acumulado Cantidad de Materias"] = muestras["Cantidad de Materias"].cummax()
    muestras["Max Acumulado Cantidad de Grupos"] = muestras["Cantidad de Grupos"].cummax()

    # Campos temporales para hacerle seguimiento a la primera occurencia de maximos locales
    mask_materias = (muestras["Cantidad de Materias"] == muestras["Max Acumulado Cantidad de Materias"])
    mask_grupos = (muestras["Cantidad de Grupos"] == muestras["Max Acumulado Cantidad de Grupos"])
    muestras["last_max_idx_materias"] = muestras["index"].where(mask_materias).ffill().astype(int)
    muestras["last_max_idx_grupos"] = muestras["index"].where(mask_grupos).ffill().astype(int)

    muestras["Cantidad de Semestres sin Sobrecarga de Asignaturas"] = muestras["index"] - muestras["last_max_idx_materias"]
    muestras["Cantidad de Semestres sin Sobrecarga de Grupos"] = muestras["index"] - muestras["last_max_idx_grupos"]

    muestras["Indice de Carga Asignaturas"] = muestras["Cantidad de Materias"] / muestras["Max Acumulado Cantidad de Materias"]
    muestras["Indice de Carga Grupos"] = muestras["Cantidad de Grupos"] / muestras["Max Acumulado Cantidad de Grupos"]
    muestras["Indice de Carga Académica"] = (muestras["Indice de Carga Asignaturas"] + muestras["Indice de Carga Grupos"]) / 2

    muestras.drop(columns=["index", "Max Acumulado Cantidad de Materias", "Max Acumulado Cantidad de Grupos", "last_max_idx_materias", "last_max_idx_grupos"], inplace=True)
    muestras.set_index(["id_docente"], inplace=True)
    return muestras


In [12]:
# Se genera el KPI de Versatilidad Docente y Carga Academica

cantidad_de_materias_que_el_docente_maneja_por_cada_semestre = (
    cantidad_de_materias_que_el_docente_maneja_por_cada_semestre
    .set_index(["id_docente"], append=False)
    .groupby(level=["id_docente"], group_keys=False)
    .apply(agregar_metrica_carga_academica, include_groups=False)
    .reset_index()
)
cantidad_de_materias_que_el_docente_maneja_por_cada_semestre

,id_docente,semestre,Cantidad de Materias,Cantidad de Grupos,numero_estudiantes,Cantidad de Semestres sin Sobrecarga de Asignaturas,Cantidad de Semestres sin Sobrecarga de Grupos,Indice de Carga Asignaturas,Indice de Carga Grupos,Indice de Carga Académica
0,Docente_1,2020Q1,5,7,205,0,0,1.0,1.0,1.0
1,Docente_1,2020Q3,6,9,303,0,0,1.0,1.0,1.0
2,Docente_1,2021Q1,5,9,208,1,0,0.833333,1.0,0.916667
3,Docente_1,2021Q3,5,7,182,2,1,0.833333,0.777778,0.805556
4,Docente_1,2022Q1,3,6,195,3,2,0.5,0.666667,0.583333
...,...,...,...,...,...,...,...,...,...,...
395,Docente_9,2021Q3,5,6,199,3,3,0.833333,0.666667,0.75
396,Docente_9,2022Q1,6,9,305,0,0,1.0,1.0,1.0
397,Docente_9,2022Q3,5,8,281,1,1,0.833333,0.888889,0.861111
398,Docente_9,2023Q1,3,4,101,2,2,0.5,0.444444,0.472222


## Seguimiento de Asignaturas

In [13]:

asignaturas_historico = (
    df.groupby(["asignatura", "semestre"], as_index=False)
    .agg(
        Cantidad_de_Docentes=("id_docente", "nunique"),
        Cantidad_de_Grupos=("semestre", "size"), # Counts rows per group
        numero_estudiantes=("numero_estudiantes", "sum"),
    )
    .rename(columns={"Cantidad_de_Grupos": "Cantidad de Grupos", "Cantidad_de_Docentes": "Cantidad de Docentes"})
    .sort_values(by=["asignatura", "semestre"], ascending=[True, True])
)


asignaturas_historico["Cantidad de Docentes"] = asignaturas_historico["Cantidad de Docentes"].astype("Int64")
asignaturas_historico["Cantidad de Grupos"] = asignaturas_historico["Cantidad de Grupos"].astype("Int64")
asignaturas_historico["numero_estudiantes"] = asignaturas_historico["numero_estudiantes"].astype("Int64")

asignaturas_historico


,asignatura,semestre,Cantidad de Docentes,Cantidad de Grupos,numero_estudiantes
0,Bases de Datos,2020Q1,31,56,1558
1,Bases de Datos,2020Q3,35,60,1739
2,Bases de Datos,2021Q1,34,58,1768
3,Bases de Datos,2021Q3,33,48,1405
4,Bases de Datos,2022Q1,36,48,1402
5,Bases de Datos,2022Q3,28,51,1588
6,Bases de Datos,2023Q1,35,60,1768
7,Bases de Datos,2023Q3,32,47,1474
8,Estructuras de Datos,2020Q1,36,66,2038
9,Estructuras de Datos,2020Q3,30,54,1558


In [14]:
# df["Asistencia Promedia de los Estudiantes"] = np.random.randint(0, 100, len(df))